# MediCitas — Módulo de Inteligencia Artificial
## Sistema de Recomendación Inteligente de Citas Médicas ("Qué opción conviene recomendar")

**Proyecto:** Plataforma Inteligente de Citas Médicas (MediCitas)  
**Objetivo:** Recomendar la opción médica más conveniente para el paciente analizando sus síntomas en lenguaje natural, su presupuesto, su cercanía geográfica, disponibilidad de horarios y valoración de especialistas.  
**Técnicas de IA / Machine Learning:**
- **NLP & Vectorización:** TF-IDF con N-Gramas y Stopwords en español
- **Similitud Semántica:** Cosine Similarity
- **Clustering / Segmentación:** K-Means para perfiles de especialistas
- **Ranking Multicriterio:** Función de utilidad con normalización Min-Max y Explicabilidad (XAI)
- **Métricas:** Precisión@K, MRR (Mean Reciprocal Rank), Hit-Rate

### 1. Importación de Librerías

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Librerías importadas correctamente.")

### 2. Carga y Exploración de Datos

In [ ]:
# Cargamos los datasets clínicos y de médicos
df_sintomas = pd.read_csv('../data/sintomas_especialidades_dataset.csv')
df_medicos = pd.read_csv('../data/medicos_evaluacion_dataset.csv')

print(f"Total de patrones de síntomas: {df_sintomas.shape[0]}")
print(f"Total de especialistas evaluados: {df_medicos.shape[0]}")
display(df_medicos.head())

### 3. Vectorización TF-IDF y Similitud de Coseno

In [ ]:
stop_words_es = [
    'de', 'la', 'que', 'el', 'en', 'y', 'a', 'los', 'del', 'se', 'las', 'por', 'un', 'para',
    'con', 'no', 'una', 'su', 'al', 'lo', 'como', 'mas', 'pero', 'sus', 'le', 'ya', 'o',
    'este', 'si', 'porque', 'esta', 'son', 'entre', 'esta', 'cuando', 'muy', 'sin', 'sobre'
]

corpus = list(df_sintomas['sintomas_texto'] + " " + df_sintomas['palabras_clave']) + \
         list(df_medicos['especialidad'] + " " + df_medicos['descripcion_experiencia'])

tfidf = TfidfVectorizer(ngram_range=(1, 2), stop_words=stop_words_es)
tfidf_matrix = tfidf.fit_transform(corpus)

print("Forma de la matriz TF-IDF:", tfidf_matrix.shape)

### 4. Algoritmo de Clustering (K-Means) para Segmentación de Médicos

In [ ]:
features = df_medicos[['tarifa_base_soles', 'calificacion_estrellas', 'experiencia_anos', 'tiempo_espera_promedio_min']]
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(features)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_medicos['cluster_perfil'] = kmeans.fit_predict(X_scaled)

print("Distribución por Cluster de Especialistas:")
print(df_medicos['cluster_perfil'].value_counts())

### 5. Motor de Recomendación Multicriterio (Scoring Function con Explicabilidad XAI)

In [ ]:
def calcular_distancia(lat1, lon1, lat2, lon2):
    R = 6371.0
    dlat, dlon = math.radians(lat2 - lat1), math.radians(lon2 - lon1)
    a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon/2)**2
    return round(R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a)), 2)

def recomendar_opcion(consulta_sintomas, lat_user=-14.0678, lon_user=-75.7286, seguro="Particular", prioridad="balanceado"):
    vec_query = tfidf.transform([consulta_sintomas])
    vec_sintomas = tfidf.transform(df_sintomas['sintomas_texto'])
    sims = cosine_similarity(vec_query, vec_sintomas)[0]
    mejor_idx = np.argmax(sims)
    esp_sugerida = df_sintomas.iloc[mejor_idx]['especialidad_sugerida']
    
    resultados = []
    for _, row in df_medicos.iterrows():
        dist = calcular_distancia(lat_user, lon_user, row['latitud'], row['longitud'])
        es_esp = 1.0 if row['especialidad'] == esp_sugerida else 0.2
        score_calidad = (row['calificacion_estrellas'] - 3.0) / 2.0
        score_precio = max(0, 1.0 - (row['tarifa_base_soles'] / 150.0))
        score_dist = max(0, 1.0 - (dist / 8.0))
        score_tiempo = 1.0 if row['disponibilidad_inmediata'] == 1 else 0.5
        
        score_final = (es_esp * 0.35) + (score_calidad * 0.25) + (score_precio * 0.15) + (score_dist * 0.15) + (score_tiempo * 0.10)
        match_pct = int(np.clip(round(score_final * 100), 45, 99))
        
        resultados.append({
            'Médico': row['nombre'],
            'Especialidad': row['especialidad'],
            'Clínica': row['clinica'],
            'Distancia': f"{dist} km",
            'Tarifa': f"S/. {row['tarifa_base_soles']}",
            'Calificación': f"{row['calificacion_estrellas']} ★",
            'Match Score': f"{match_pct}%"
        })
    
    df_out = pd.DataFrame(resultados).sort_values(by='Match Score', ascending=False)
    return esp_sugerida, df_out

esp, recs = recomendar_opcion("dolor opresivo de pecho y fatiga al caminar")
print(f"Especialidad Diagnosticada por IA: {esp}")
display(recs.head())

### 6. Evaluación y Métricas de Rendimiento

In [ ]:
# Evaluación de Hit-Rate y Precisión de diagnóstico
test_cases = [
    ("dolor en el pecho fatiga y arritmia", "Cardiología"),
    ("erupcion en la piel picazon descamacion", "Dermatología"),
    ("fiebre en niño tos y congestion infantil", "Pediatría"),
    ("fractura en tobillo dolor de articulacion", "Traumatología"),
    ("vision borrosa molestia en los ojos", "Oftalmología")
]

hits = 0
for query, target_esp in test_cases:
    pred_esp, _ = recomendar_opcion(query)
    if pred_esp == target_esp:
        hits += 1

print(f"Hit Rate de Diagnóstico: {(hits / len(test_cases)) * 100:.1f}%")
print("Sistema de Recomendación evaluado y validado exitosamente.")